# 🔍 Buscador de Empresas — SRI & SCVS

Busca cualquier empresa en los datos del SRI y Superintendencia de Compañías.

**Modos disponibles:**
- **Búsqueda individual**: ingresa un nombre manualmente
- **Búsqueda masiva**: sube un archivo CSV/Excel con hasta 441+ empresas → retorna top 3 coincidencias por empresa

In [ ]:
# ── Instalación de dependencias ────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "rapidfuzz", "-q"], check=False)
print("✓ Dependencias listas")

In [ ]:
# ── Imports y configuración de rutas ──────────────────────────────────────────
import os, glob, unicodedata, warnings
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

warnings.filterwarnings("ignore")

BASE_DIR = os.path.abspath(".")  # Carpeta raíz del proyecto
SRI_DIR  = os.path.join(BASE_DIR, "02_data_cleaning", "data_SRI")
SCVS_DIR = os.path.join(BASE_DIR, "02_data_cleaning", "data_super_compañias")

try:
    from rapidfuzz import fuzz, process as rfprocess
    USE_RAPIDFUZZ = True
    print("✓ Motor: rapidfuzz (token_set_ratio)")
except ImportError:
    from difflib import SequenceMatcher
    USE_RAPIDFUZZ = False
    print("⚠ Motor: difflib (instale rapidfuzz para mejor rendimiento)")

print(f"✓ SRI  → {SRI_DIR}")
print(f"✓ SCVS → {SCVS_DIR}")

In [ ]:
# ── Funciones core ─────────────────────────────────────────────────────────────

def normalize(text) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    text = text.upper().strip()
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    return text


def scores_for_series(query_norm: str, series: pd.Series) -> pd.Series:
    norm_series = series.map(normalize)
    if USE_RAPIDFUZZ:
        names = norm_series.tolist()
        results = rfprocess.cdist(
            [query_norm], names, scorer=fuzz.token_set_ratio, workers=1
        )[0]
        return pd.Series(results, index=series.index)
    else:
        return norm_series.map(
            lambda x: SequenceMatcher(None, query_norm, x).ratio() * 100 if x else 0.0
        )


def search_sri(query_norm: str, threshold: int = 60, top_n: int = 3) -> list:
    """Busca en todos los CSV del SRI y retorna lista ordenada por score."""
    results = []
    csv_files = sorted(glob.glob(os.path.join(SRI_DIR, "SRI_RUC_*.csv")))

    for csv_path in csv_files:
        fname = os.path.basename(csv_path)
        try:
            with open(csv_path, "r", encoding="latin-1") as fh:
                header_line = fh.readline().strip()
            all_cols = header_line.split("|")

            if "RAZON_SOCIAL" not in all_cols:
                continue

            fantasia_col = next(
                (c for c in all_cols if "FANTASIA" in c.upper()), None
            )
            keep_cols = ["NUMERO_RUC", "RAZON_SOCIAL", "ESTADO_CONTRIBUYENTE"]
            if fantasia_col:
                keep_cols.append(fantasia_col)
            usecols = [c for c in keep_cols if c in all_cols]

            for chunk in pd.read_csv(
                csv_path,
                sep="|",
                encoding="latin-1",
                usecols=usecols,
                on_bad_lines="skip",
                chunksize=50_000,
                dtype=str,
                low_memory=False,
            ):
                chunk = chunk.fillna("")
                sc_rs = scores_for_series(query_norm, chunk["RAZON_SOCIAL"])
                mask = sc_rs >= threshold

                sc_fn = pd.Series(0.0, index=chunk.index)
                if fantasia_col and fantasia_col in chunk.columns:
                    sc_fn = scores_for_series(query_norm, chunk[fantasia_col])
                    mask = mask | (sc_fn >= threshold)

                hits = chunk[mask].copy()
                if hits.empty:
                    continue

                for idx, row in hits.iterrows():
                    rs = row.get("RAZON_SOCIAL", "")
                    fn = row.get(fantasia_col, "") if fantasia_col else ""
                    s_rs = float(sc_rs.get(idx, 0))
                    s_fn = float(sc_fn.get(idx, 0)) if fantasia_col else 0.0
                    best = max(s_rs, s_fn)
                    matched = "RAZON_SOCIAL" if s_rs >= s_fn else fantasia_col

                    results.append({
                        "fuente": f"SRI",
                        "archivo": fname,
                        "ruc": row.get("NUMERO_RUC", ""),
                        "razon_social": rs,
                        "nombre_fantasia": fn,
                        "estado": row.get("ESTADO_CONTRIBUYENTE", ""),
                        "score": round(best, 1),
                        "columna_match": matched,
                    })
        except Exception:
            pass

    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_n]


def search_scvs(query_norm: str, threshold: int = 60, top_n: int = 3) -> list:
    """Busca en todos los XLSX del SCVS (excluye ranking) y retorna lista ordenada."""
    results = []
    xlsx_files = sorted([
        f for f in glob.glob(os.path.join(SCVS_DIR, "*.xlsx"))
        if "ranking" not in os.path.basename(f).lower()
    ])

    for xlsx_path in xlsx_files:
        fname = os.path.basename(xlsx_path)
        for header_row in range(8):
            try:
                df = pd.read_excel(xlsx_path, header=header_row, dtype=str)
                df = df.fillna("")
                named_cols = [c for c in df.columns if not str(c).startswith("Unnamed")]
                if len(named_cols) < 3 or "NOMBRE" not in df.columns:
                    continue

                sc = scores_for_series(query_norm, df["NOMBRE"])
                mask = sc >= threshold
                hits = df[mask].copy()

                id_col = next(
                    (c for c in ["RUC", "IDENTIFICACIÓN", "IDENTIFICACION", "EXPEDIENTE"]
                     if c in df.columns), None
                )

                for idx, row in hits.iterrows():
                    nombre = row.get("NOMBRE", "")
                    s = float(sc.get(idx, 0))
                    detail_keys = [
                        c for c in df.columns
                        if c not in ["NOMBRE", "No. FILA", "!CODIGO!"]
                        and not str(c).startswith("Unnamed")
                        and row.get(c, "")
                    ]
                    detalles = {k: row[k] for k in detail_keys[:8]}

                    results.append({
                        "fuente": "SCVS",
                        "archivo": fname,
                        "id": row.get(id_col, "") if id_col else "",
                        "nombre": nombre,
                        "score": round(s, 1),
                        "columna_match": "NOMBRE",
                        "detalles": detalles,
                    })
                break
            except Exception:
                continue

    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_n]


def buscar(query: str, threshold: int = 75, top_n: int = 3) -> dict:
    """Punto de entrada unificado. Retorna {sri: [...], scvs: [...]}"""
    query_norm = normalize(query)
    return {
        "sri":  search_sri(query_norm,  threshold, top_n),
        "scvs": search_scvs(query_norm, threshold, top_n),
    }

print("✓ Funciones cargadas")

---
## Modo 1 — Búsqueda individual
Ingresa un nombre y ve el resultado directamente.

In [ ]:
# ── Parámetros búsqueda individual ────────────────────────────────────────────
EMPRESA_BUSCAR = "CORPORACION FAVORITA"   # <-- cambia aquí
UMBRAL         = 75                        # Score mínimo 0-100
TOP_N          = 3                         # Top resultados por fuente

In [ ]:
# ── Ejecutar búsqueda individual ───────────────────────────────────────────────
print(f'Buscando: "{EMPRESA_BUSCAR}"  |  umbral={UMBRAL}%  |  top={TOP_N}')
print("Espere...\n")

resultados = buscar(EMPRESA_BUSCAR, threshold=UMBRAL, top_n=TOP_N)

# ── Mostrar SRI ──
sri_rows = []
for r in resultados["sri"]:
    sri_rows.append({
        "Score %": r["score"],
        "RUC": r["ruc"],
        "Razón Social": r["razon_social"],
        "Nombre Comercial": r["nombre_fantasia"],
        "Estado": r["estado"],
        "Match en": r["columna_match"],
        "Archivo": r["archivo"],
    })

print(f"═══ SRI — {len(resultados['sri'])} resultado(s) ═══")
if sri_rows:
    display(pd.DataFrame(sri_rows))
else:
    print("  Sin resultados.")

# ── Mostrar SCVS ──
scvs_rows = []
for r in resultados["scvs"]:
    row = {"Score %": r["score"], "ID/RUC": r["id"], "Nombre": r["nombre"]}
    row.update(r.get("detalles", {}))
    row["Archivo"] = r["archivo"]
    scvs_rows.append(row)

print(f"\n═══ SCVS — {len(resultados['scvs'])} resultado(s) ═══")
if scvs_rows:
    display(pd.DataFrame(scvs_rows))
else:
    print("  Sin resultados.")

---
## Modo 2 — Búsqueda masiva (archivo con lista de empresas)

Sube un archivo **CSV o Excel** que tenga una columna con los nombres de empresa.  
El script detecta automáticamente la columna de nombres o puedes especificarla.

**Resultado**: un DataFrame/Excel con top 3 coincidencias (SRI + SCVS) por cada empresa.

In [ ]:
# ── Parámetros búsqueda masiva ─────────────────────────────────────────────────

# Ruta al archivo con la lista de empresas (CSV o Excel)
# Déjalo en None para usar el widget de carga interactiva
ARCHIVO_EMPRESAS  = None   # Ejemplo: r"C:\Users\...\mis_empresas.csv"

# Nombre de la columna que contiene los nombres de empresa
# Si es None se intentará detectar automáticamente
COLUMNA_NOMBRE    = None   # Ejemplo: "nombre_empresa"

UMBRAL_MASIVO     = 75     # Score mínimo 0-100
TOP_N_MASIVO      = 3      # Top coincidencias por empresa
ARCHIVO_SALIDA    = "resultados_busqueda_empresas.xlsx"  # Archivo de salida

In [ ]:
# ── Widget de carga si no se especificó ruta ───────────────────────────────────
import io

_uploaded_file = None

if ARCHIVO_EMPRESAS is None:
    uploader = widgets.FileUpload(
        accept='.csv,.xlsx,.xls',
        multiple=False,
        description='Subir archivo',
        layout=widgets.Layout(width='300px')
    )
    btn_confirm = widgets.Button(
        description="Confirmar archivo",
        button_style="success",
        layout=widgets.Layout(width='200px')
    )
    out_info = widgets.Output()

    def on_confirm(b):
        global _uploaded_file
        with out_info:
            clear_output()
            if not uploader.value:
                print("⚠ Ningún archivo seleccionado.")
                return
            uploaded = list(uploader.value.values())[0]
            _uploaded_file = {
                "name": uploaded["metadata"]["name"],
                "content": uploaded["content"],
            }
            print(f"✓ Archivo cargado: {_uploaded_file['name']}")

    btn_confirm.on_click(on_confirm)
    display(widgets.VBox([uploader, btn_confirm, out_info]))
else:
    print(f"✓ Archivo configurado: {ARCHIVO_EMPRESAS}")

In [ ]:
# ── Leer archivo de empresas ───────────────────────────────────────────────────

def leer_archivo_empresas(ruta=None, contenido_bytes=None, nombre_archivo=None):
    """Lee CSV o Excel desde ruta o desde bytes en memoria."""
    if contenido_bytes is not None:
        buf = io.BytesIO(bytes(contenido_bytes))
        ext = os.path.splitext(nombre_archivo or "")[-1].lower()
    else:
        buf = ruta
        ext = os.path.splitext(ruta or "")[-1].lower()

    if ext in (".xlsx", ".xls"):
        return pd.read_excel(buf, dtype=str)
    else:
        # Intentar auto-detectar separador
        for sep in [",", ";", "\t", "|"]:
            try:
                df = pd.read_csv(buf if contenido_bytes is None else io.BytesIO(bytes(contenido_bytes)),
                                 sep=sep, encoding="utf-8", dtype=str, nrows=5)
                if len(df.columns) >= 1:
                    return pd.read_csv(
                        buf if contenido_bytes is None else io.BytesIO(bytes(contenido_bytes)),
                        sep=sep, encoding="utf-8", dtype=str
                    )
            except Exception:
                try:
                    df = pd.read_csv(buf if contenido_bytes is None else io.BytesIO(bytes(contenido_bytes)),
                                     sep=sep, encoding="latin-1", dtype=str, nrows=5)
                    if len(df.columns) >= 1:
                        return pd.read_csv(
                            buf if contenido_bytes is None else io.BytesIO(bytes(contenido_bytes)),
                            sep=sep, encoding="latin-1", dtype=str
                        )
                except Exception:
                    pass
    raise ValueError("No se pudo leer el archivo. Verifique el formato.")


def detectar_columna_nombre(df: pd.DataFrame) -> str:
    """Heurística: busca columna con palabras clave de nombre de empresa."""
    keywords = ["nombre", "name", "empresa", "compania", "compañia",
                "razon", "razon_social", "denominacion"]
    for col in df.columns:
        col_norm = normalize(col)
        if any(kw in col_norm for kw in keywords):
            return col
    # Fallback: primera columna de texto con variedad suficiente
    for col in df.columns:
        if df[col].dropna().nunique() > 5:
            return col
    return df.columns[0]


# Cargar el DataFrame de empresas
if ARCHIVO_EMPRESAS is not None:
    df_empresas = leer_archivo_empresas(ruta=ARCHIVO_EMPRESAS)
elif _uploaded_file is not None:
    df_empresas = leer_archivo_empresas(
        contenido_bytes=_uploaded_file["content"],
        nombre_archivo=_uploaded_file["name"]
    )
else:
    raise ValueError("⚠ Debes subir un archivo o configurar ARCHIVO_EMPRESAS antes de ejecutar esta celda.")

# Detectar columna de nombre
col_nombre = COLUMNA_NOMBRE if COLUMNA_NOMBRE else detectar_columna_nombre(df_empresas)

print(f"✓ Archivo cargado: {len(df_empresas)} filas")
print(f"✓ Columna de nombre detectada: '{col_nombre}'")
print(f"✓ Otras columnas: {[c for c in df_empresas.columns if c != col_nombre]}")
print()
print("Vista previa:")
display(df_empresas[[col_nombre]].head(10))

In [ ]:
# ── Búsqueda masiva ────────────────────────────────────────────────────────────
from tqdm.notebook import tqdm

empresas = df_empresas[col_nombre].dropna().str.strip().unique().tolist()
total = len(empresas)
print(f"Total de empresas únicas a buscar: {total}")
print(f"Umbral: {UMBRAL_MASIVO}%  |  Top por empresa: {TOP_N_MASIVO}")
print()

filas_resultado = []

for empresa in tqdm(empresas, desc="Buscando empresas"):
    if not empresa:
        continue
    res = buscar(empresa, threshold=UMBRAL_MASIVO, top_n=TOP_N_MASIVO)

    # Top 3 SRI
    for rank, r in enumerate(res["sri"], 1):
        filas_resultado.append({
            "empresa_input": empresa,
            "fuente": "SRI",
            "rank": rank,
            "score_%": r["score"],
            "ruc": r["ruc"],
            "razon_social": r["razon_social"],
            "nombre_fantasia": r["nombre_fantasia"],
            "estado": r["estado"],
            "columna_match": r["columna_match"],
            "archivo_fuente": r["archivo"],
        })

    # Top 3 SCVS
    for rank, r in enumerate(res["scvs"], 1):
        det = r.get("detalles", {})
        filas_resultado.append({
            "empresa_input": empresa,
            "fuente": "SCVS",
            "rank": rank,
            "score_%": r["score"],
            "ruc": r["id"],
            "razon_social": r["nombre"],
            "nombre_fantasia": "",
            "estado": det.get("SITUACIÓN LEGAL", det.get("ESTADO", "")),
            "columna_match": "NOMBRE",
            "archivo_fuente": r["archivo"],
            **{f"scvs_{k}": v for k, v in list(det.items())[:5]},
        })

    # Si no hay resultados en ninguna fuente
    if not res["sri"] and not res["scvs"]:
        filas_resultado.append({
            "empresa_input": empresa,
            "fuente": "SIN MATCH",
            "rank": None,
            "score_%": None,
            "razon_social": "",
        })

df_resultado = pd.DataFrame(filas_resultado)
print(f"\n✓ Búsqueda completada: {len(df_resultado)} filas de resultados")
display(df_resultado.head(20))

In [ ]:
# ── Guardar resultados en Excel ────────────────────────────────────────────────
output_path = os.path.join(BASE_DIR, ARCHIVO_SALIDA)

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    # Hoja 1: resultados completos
    df_resultado.to_excel(writer, sheet_name="Resultados", index=False)

    # Hoja 2: resumen — mejor match por empresa y fuente
    resumen = (
        df_resultado[df_resultado["rank"] == 1]
        .groupby(["empresa_input", "fuente"])[["score_%", "ruc", "razon_social", "estado"]]
        .first()
        .reset_index()
    )
    resumen.to_excel(writer, sheet_name="Resumen_Top1", index=False)

    # Hoja 3: empresas sin match
    sin_match = df_resultado[df_resultado["fuente"] == "SIN MATCH"][["empresa_input"]]
    sin_match.to_excel(writer, sheet_name="Sin_Match", index=False)

print(f"✓ Resultados guardados en: {output_path}")
print(f"  - Hoja 'Resultados'   : {len(df_resultado)} filas")
print(f"  - Hoja 'Resumen_Top1' : {len(resumen)} filas")
print(f"  - Hoja 'Sin_Match'    : {len(sin_match)} filas")